In [9]:
import qiskit
print(qiskit.__version__)

2.4.1


In [10]:
###### Algo de Deutsch-Jozsa ######
## Version de base avec f : Z/2Z^(2) -> Z/2Z

In [11]:
from qiskit import QuantumCircuit
import itertools

In [12]:
#Création procédurale de toutes les fonctions pour (Z/2Z^n) -> Z/2Z, en fonction de n

def generate_oracles(n):
    num_rows = 2**n
    num_oracles = 2**num_rows

    print(f"Generating {num_oracles} oracles for {num_rows} input combinations.")

    oracle_list = []

    for k, truth_table_outputs in enumerate(itertools.product([0,1], repeat = num_rows)):
        qc = QuantumCircuit(n + 1, name=f"Oracle_f_{k}")
        target = n  # The target qubit is the last one

        #Now, truth_table_outputs = (x0, x1, ..., x_{2^n - 1}) where each x_i is either 0 or 1
        for integer_input, result in enumerate(truth_table_outputs):
            if result == 1: #Il faut s'assurer que le mcx tire quand l'output attendu est 1 (vrai donc))
                # Ici on passe par la représentation binaire de l'index de la ligne pour passer l'entrée associée à cette ligne de la sortie
                # Convert the row index to a binary string padded to n bits
                binary_string = format(integer_input, f'0{n}b')

                #Attention à respecter la convention adoptée par Qiskit
                #J'inverse les bits à 0 pour faire tirer le mcx quand il faut inverser l'ancille
                for bit_index, char in enumerate(reversed(binary_string)):
                    if char == '0':
                        qc.x(bit_index)

                qc.mcx(list(range(n)), target)

                #Je rétablis les bits originaux
                for bit_index, char in enumerate(reversed(binary_string)):
                    if char == '0':
                        qc.x(bit_index)

        oracle_list.append(qc)
    
    return oracle_list

In [13]:
def qc_DJ(n, oracle):
    #Creation du circuit de Deutsch-Jozsa
    qc = QuantumCircuit(n + 1, n)

    qc.h(list(range(n)))  # Apply Hadamard gates to the first n qubits

    qc.x(n)
    qc.h(n)
    
    qc.barrier()

    qc.compose(oracle, inplace=True)

    qc.barrier()

    qc.h(list(range(n)))

    qc.measure(range(n), range(n))

    return qc

In [14]:
def categorize_function(n, counts):
    """
    Classifie une fonction à partir des résultats de mesure du circuit DJ.
    - constante  : 100% des shots tombent sur '0...0'
    - equilibre  : 0% des shots tombent sur '0...0'
    - other      : proportion intermédiaire (ni constante ni équilibrée)
    """
    all_zeros = '0' * n
    total_shots = sum(counts.values())
    zeros_count = counts.get(all_zeros, 0)

    if zeros_count == total_shots:
        return "constante"
    elif zeros_count == 0:
        return "equilibre"
    else:
        return "other"

In [15]:
from qiskit_aer.backends import AerSimulator
from qiskit import transpile

def run_DJ_circuit_on_oracle(n, qc_DJ, oracle):
    backend = AerSimulator()

    qc = qc_DJ(n, oracle)
    transpiled_qc = transpile(qc, backend)
    job = backend.run(transpiled_qc, shots=1024)
    counts = job.result().get_counts(transpiled_qc)

    category = categorize_function(n, counts) 
    return category

In [16]:
import pandas as pd

N_MAX = 3
n = N_MAX

final_table = {}

for n in range (1, N_MAX + 1):
    oracles = generate_oracles(n)
    print(f"Generated {len(oracles)} oracles for n={n}.")

    counts = {
        "const": 0,
        "equilib": 0,
        "others": 0,
        "somme": 0,
        "attendu": 2**(2**n)
    }

    for oracle in oracles:
        category = run_DJ_circuit_on_oracle(n, qc_DJ, oracle)
        
        # 2. On incrémente le bon compteur en fonction du retour
        if category == "constante":
            counts["const"] += 1
        elif category == "equilibre":
            counts["equilib"] += 1
        else:
            counts["others"] += 1

        counts["somme"] += 1
    
    final_table[f"n={n}"] = counts

df_results = pd.DataFrame(final_table)

# Affichage du résultat
print("\n=== GRILLE DE SYNTHÈSE DE L'EXPÉRIENCE ===")
print(df_results)

Generating 4 oracles for 2 input combinations.
Generated 4 oracles for n=1.
Generating 16 oracles for 4 input combinations.
Generated 16 oracles for n=2.
Generating 256 oracles for 8 input combinations.
Generated 256 oracles for n=3.

=== GRILLE DE SYNTHÈSE DE L'EXPÉRIENCE ===
         n=1  n=2  n=3
const      2    2    2
equilib    2    6   70
others     0    8  184
somme      4   16  256
attendu    4   16  256
